<a href="https://colab.research.google.com/github/Joacco11/TelecomX_parte2_Latam/blob/main/TelecomX_parte2_Latam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Telecom X – Parte 2: Predicción de Cancelación (Churn)

##🛠️ Preparación de los Datos

###Extracción del Archivo Tratado

In [ ]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import RandomOverSampler
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
url = 'https://raw.githubusercontent.com/Joacco11/TelecomX_Latam-Part-1/refs/heads/main/telecomx_clean.csv'
df = pd.read_csv(url, index_col="id_cliente")
df.head()

,churn,genero,adulto_mayor,pareja,dependientes,antiguedad_meses,servicio_telefonico,lineas_multiples,servicio_internet,seguridad_en_linea,...,proteccion_dispositivo,soporte_tecnico,streaming_tv,streaming_peliculas,tipo_contrato,facturacion_electronica,metodo_pago,cargo_mensual,cargo_total,Cuentas_Diarias
id_cliente,,,,,,,,,,,,,,,,,,,,,
0002-ORFBO,No,Female,False,True,True,9,True,No,DSL,No,...,No,Yes,Yes,No,One year,True,Mailed check,65.6,593.30,2.19
0003-MKNFE,No,Male,False,False,False,9,True,Yes,DSL,No,...,No,No,No,Yes,Month-to-month,False,Mailed check,59.9,542.40,2.00
0004-TLHLJ,Yes,Male,False,False,False,4,True,No,Fiber optic,No,...,Yes,No,No,No,Month-to-month,True,Electronic check,73.9,280.85,2.46
0011-IGKFF,Yes,Male,True,True,False,13,True,No,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,True,Electronic check,98.0,1237.85,3.27
0013-EXCHZ,Yes,Female,True,True,False,3,True,No,Fiber optic,No,...,No,Yes,Yes,No,Month-to-month,True,Mailed check,83.9,267.40,2.80


>En el proyecto anterior no eliminé los valores nulos en **churn** y **cargo_total** porque lo que buscaba era representar la realidad del negocio, pero en el caso del desarrollo del modelo predictivo, realizaré la limpieza de estos campos ya que no los necesito

In [ ]:
# 1. Eliminar filas con "No se sabe" en churn
df = df[df['churn'] != "No se sabe"]

# 2. Imputar valores nulos en cargo_total con la media
df['cargo_total'] = df['cargo_total'].fillna(df['cargo_total'].mean())

# 3. Verificar
print(df['churn'].value_counts())
print("Nulos en cargo_total:", df['cargo_total'].isna().sum())

churn
No     5174
Yes    1869
Name: count, dtype: int64
Nulos en cargo_total: 0



###Eliminación de Columnas Irrelevantes

>Id_cliente está como index. Cuentas_diarias no la  al ser una columna de caracter artificial.


In [ ]:
df = df.drop(columns=['Cuentas_Diarias'])
df.shape

(7043, 20)

###Encoding

####Separación de target y variables

In [ ]:
x = df.drop(columns=['churn'])
y = df['churn']

x.dtypes

,0
genero,object
adulto_mayor,bool
pareja,bool
dependientes,bool
antiguedad_meses,int64
servicio_telefonico,bool
lineas_multiples,object
servicio_internet,object
seguridad_en_linea,object
respaldo_en_linea,object


In [ ]:
cat = x.select_dtypes(include=['object']).columns
num = x.select_dtypes(include=['int', 'float']).columns
bin = x.select_dtypes(include=['bool']).columns

print("Categóricas:", cat.tolist())
print("Numéricas:", num.tolist())
print("Booleanas:", bin.tolist())

Categóricas: ['genero', 'lineas_multiples', 'servicio_internet', 'seguridad_en_linea', 'respaldo_en_linea', 'proteccion_dispositivo', 'soporte_tecnico', 'streaming_tv', 'streaming_peliculas', 'tipo_contrato', 'metodo_pago']
Numéricas: ['antiguedad_meses', 'cargo_mensual', 'cargo_total']
Booleanas: ['adulto_mayor', 'pareja', 'dependientes', 'servicio_telefonico', 'facturacion_electronica']


####Tratamiento de categóricas


In [ ]:
one_hot =  make_column_transformer((OneHotEncoder(drop='first'), cat), remainder='passthrough',
                                   sparse_threshold=0,force_int_remainder_cols=False)
x_enc = one_hot.fit_transform(x)
ohe_cols = one_hot.named_transformers_['onehotencoder'].get_feature_names_out(cat)
# Combinar nombres de OHE + resto de columnas
all_cols = list(ohe_cols) + [col for col in x.columns if col not in cat]

# Convertir a DataFrame
x = pd.DataFrame(x_enc, columns=all_cols, index=x.index)

print(x.shape)
print(all_cols)

(7043, 30)
['genero_Male', 'lineas_multiples_No phone service', 'lineas_multiples_Yes', 'servicio_internet_Fiber optic', 'servicio_internet_No', 'seguridad_en_linea_No internet service', 'seguridad_en_linea_Yes', 'respaldo_en_linea_No internet service', 'respaldo_en_linea_Yes', 'proteccion_dispositivo_No internet service', 'proteccion_dispositivo_Yes', 'soporte_tecnico_No internet service', 'soporte_tecnico_Yes', 'streaming_tv_No internet service', 'streaming_tv_Yes', 'streaming_peliculas_No internet service', 'streaming_peliculas_Yes', 'tipo_contrato_One year', 'tipo_contrato_Two year', 'metodo_pago_Credit card (automatic)', 'metodo_pago_Electronic check', 'metodo_pago_Mailed check', 'adulto_mayor', 'pareja', 'dependientes', 'antiguedad_meses', 'servicio_telefonico', 'facturacion_electronica', 'cargo_mensual', 'cargo_total']


####Tratamiento del target


In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
y

array([0, 0, 1, ..., 0, 0, 0])

###Verificación de la Proporción de Cancelación (Churn)

In [ ]:
# Conteo de cada clase
unique, counts = np.unique(y, return_counts=True)

# Proporción de cada clase
proporcion = counts / len(y)

for u, c, p in zip(unique, counts, proporcion):
    print(f"Clase {u}: {c} clientes ({p:.2%})")


Clase 0: 5174 clientes (73.46%)
Clase 1: 1869 clientes (26.54%)


###Balanceo de Clases

In [ ]:
# Definir oversampler
oversampler = RandomOverSampler(random_state=42)

# Aplicar oversampling sobre X e y
X_res, y_res = oversampler.fit_resample(x, y)

print("Tamaño original:", len(y))
print("Tamaño balanceado:", len(y_res))

# Ver distribución después del oversampling
print(pd.Series(y_res).value_counts())


Tamaño original: 7043
Tamaño balanceado: 10348
0    5174
1    5174
Name: count, dtype: int64


In [ ]:
# Combinar X_res e y_res en un solo DataFrame para inspección
df_res = pd.concat([X_res, pd.Series(y_res, name="Churn")], axis=1)

print("Primeras filas después del oversampling:")
print(df_res.head(10))

# Ver proporción de clases
print("\nDistribución de clases después del oversampling:")
print(y_res.value_counts(normalize=True))

Primeras filas después del oversampling:
  genero_Male lineas_multiples_No phone service lineas_multiples_Yes  \
0         0.0                               0.0                  0.0   
1         1.0                               0.0                  1.0   
2         1.0                               0.0                  0.0   
3         1.0                               0.0                  0.0   
4         0.0                               0.0                  0.0   
5         0.0                               0.0                  0.0   
6         0.0                               0.0                  0.0   
7         1.0                               0.0                  1.0   
8         0.0                               0.0                  0.0   
9         0.0                               0.0                  1.0   

  servicio_internet_Fiber optic servicio_internet_No  \
0                           0.0                  0.0   
1                           0.0                  0.0  

AttributeError: 'numpy.ndarray' object has no attribute 'value_counts'

##🎯 Correlación y Selección de Variables

##🤖 Modelado Predictivo

##📋 Interpretación y Conclusiones